In [1]:
import os
import json

os.chdir('/home/smallyan/eval_agent')
os.makedirs('/net/scratch2/smallyan/filter_eval/doc_only_evaluation', exist_ok=True)

# Check GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA A100 80GB PCIe


# Consistency Evaluation - Documentation Only

**Paper:** "LLMs Process Lists with General Filter Heads" (arXiv:2510.26784v1, October 2025)

**Evaluation Mode:** Documentation-Only - judgments based solely on explicit statements in the documentation.

## 1. Mismatches and Unsupported Claims Analysis

### No Critical Mismatches Found

After thorough review of the 32-page documentation, all major claims are directly supported by documented results:

| Claim | Supporting Evidence |
|-------|---------------------|
| Filter heads encode predicates | Table 1 (causality 0.863), Figure 1 attention patterns |
| Portable across languages | Table 2a (0.775-0.957 cross-lingual) |
| Filter heads necessary | Table 3 (22.5% vs 99.6% after ablation) |
| Dual lazy/eager mechanism | Figures 5, 8, 11; Table 5 |
| Zero-shot probing works | Figure 6 (0.81 ± 0.02 accuracy) |

### Methodology Coherence

- Section 2 defines method (DCM, activation patching)
- Sections 3-5 implement as described
- Appendices provide additional validation

In [2]:
import pandas as pd

# Binary Checklist Table
print("="*80)
print("BINARY CHECKLIST - CONSISTENCY EVALUATION")
print("="*80)

checklist = {
    "ID": ["CS1", "CS2", "CS3", "CS4", "CS5"],
    "Criterion": [
        "Results vs Conclusion",
        "Plan vs Implementation",
        "Effect Size",
        "Justification",
        "Statistical Significance"
    ],
    "Status": ["PASS", "PASS", "PASS", "PASS", "PASS"]
}

df = pd.DataFrame(checklist)
print(df.to_string(index=False))
print("="*80)

BINARY CHECKLIST - CONSISTENCY EVALUATION
 ID                Criterion Status
CS1    Results vs Conclusion   PASS
CS2   Plan vs Implementation   PASS
CS3              Effect Size   PASS
CS4            Justification   PASS
CS5 Statistical Significance   PASS


## 2. Detailed Criterion Analysis

### CS1. Conclusion vs Documented Results: PASS

All conclusions are directly supported by results:
- Filter heads encode predicates → Table 1: causality 0.863
- Portable across languages → Table 2a: 0.775-0.957 
- Necessary for filtering → Table 3: 22.5% vs 99.6%
- Dual implementation → Figures 5, 8, 11; Table 5
- Zero-shot probing → Figure 6: 0.81 ± 0.02

### CS2. Plan vs Implementation: PASS

Section 2 methodology followed exactly:
- Section 2.1: Background → Applied in all experiments
- Section 2.2: Filter heads definition → Validated in Section 3
- Section 2.3: DCM localization → Used throughout
- Experiments (3.1-3.3) match stated plan precisely

### CS3. Effect Size: PASS

Non-trivial effects relative to baselines:
- Ablation: 77.5pp drop (100% to 22.5%) vs 0.4pp for random
- Causality: 0.863 vs ~0.17 random chance (5x)
- Δlogit: +9.03 vs -0.96 for random heads

### CS4. Justification: PASS

Key choices explicitly justified:
- DCM: "single filter head is often not strong enough" (Sec 2.3)
- Logits: "more direct linear relationship" (Sec 2.3)
- Averaging: "isolate predicate signal from positional bias" (Sec F)

### CS5. Statistical Significance: PASS

Uncertainty reported:
- SD: "8.2591 ± 3.352" (Sec 4), "0.81 ± 0.02" (Fig 6)
- Sample sizes: N=512 evaluation, N=1024 localization
- Baselines: Random, FV, CI heads compared (Tables 3-4)

## 3. Summary

The documentation demonstrates strong internal consistency across all five criteria:

1. **Results-Conclusions Alignment:** All claims backed by explicit quantitative results
2. **Plan-Implementation Match:** Experiments follow Section 2 methodology precisely  
3. **Substantial Effects:** Effects 5x-194x above baselines
4. **Justified Choices:** Each methodological decision explained
5. **Adequate Statistics:** SD, N, baselines provided

**Final Result: 5/5 PASS**

In [3]:
# Create and save JSON summary
consistency_eval = {
    "Checklist": {
        "CS1_Results_vs_Conclusion": "PASS",
        "CS2_Plan_vs_Implementation": "PASS",
        "CS3_Effect_Size": "PASS",
        "CS4_Justification": "PASS",
        "CS5_Statistical_Significance": "PASS"
    },
    "Rationale": {
        "CS1_Results_vs_Conclusion": "All conclusions directly supported by documented results: filter head causality (0.863, Table 1), cross-lingual portability (0.775-0.957, Table 2a), necessity demonstrated by ablation (22.5% vs 99.6%, Table 3), dual implementation shown in Figures 5/8/11 and Table 5, zero-shot probing validated (0.81±0.02, Figure 6). No conclusions contradict or exaggerate results.",
        
        "CS2_Plan_vs_Implementation": "Section 2 explicitly defines methodology: 2.1 notation, 2.2 filter heads/activation patching, 2.3 DCM with sparse masking. Experimental sections 3.1-3.3 follow this plan exactly: portability within task, across tasks, and ablation studies. Implementation details documented (1024 examples localization, 512 evaluation). Sections 4-5 extend methodology consistently.",
        
        "CS3_Effect_Size": "Effects are non-trivial relative to baselines: filter head ablation causes 77.5pp accuracy drop (100%→22.5%) vs 0.4pp for random heads (194x ratio). Causality 0.863 vs ~0.17 random chance (5x baseline). Delta-logit +9.03 for filter heads vs -0.96 for random. Cross-lingual transfer 0.775-0.957, all substantially above chance.",
        
        "CS4_Justification": "Key design choices explicitly justified: DCM used because 'patching single filter head is often not strong enough' due to backup mechanisms (Sec 2.3). Logits over probabilities for 'more direct linear relationship with influence' (Sec 2.3). Query caching before positional encoding because filter heads are 'semantic heads with minimal sensitivity to positional information' (Sec 2.2). Averaging to 'isolate predicate signal from positional bias' (Sec F).",
        
        "CS5_Statistical_Significance": "Key results report uncertainty: SD explicitly stated (8.2591±3.352 in Sec 4, 0.81±0.02 in Fig 6, 6.75±3.94 in Sec D). Sample sizes documented (512 evaluation, 1024 localization). Multiple baselines provided: random head ablation (Table 3), function vector/concept/induction heads comparison (Table 4) with equal head counts (79) for fairness. Metrics clearly defined (Equations 3-4)."
    }
}

# Save to JSON file
output_path = '/net/scratch2/smallyan/filter_eval/doc_only_evaluation/consistency_evaluation.json'
with open(output_path, 'w') as f:
    json.dump(consistency_eval, f, indent=2)

print(f"JSON saved to: {output_path}")
print("\nJSON Contents:")
print(json.dumps(consistency_eval, indent=2))

JSON saved to: /net/scratch2/smallyan/filter_eval/doc_only_evaluation/consistency_evaluation.json

JSON Contents:
{
  "Checklist": {
    "CS1_Results_vs_Conclusion": "PASS",
    "CS2_Plan_vs_Implementation": "PASS",
    "CS3_Effect_Size": "PASS",
    "CS4_Justification": "PASS",
    "CS5_Statistical_Significance": "PASS"
  },
  "Rationale": {
    "CS1_Results_vs_Conclusion": "All conclusions directly supported by documented results: filter head causality (0.863, Table 1), cross-lingual portability (0.775-0.957, Table 2a), necessity demonstrated by ablation (22.5% vs 99.6%, Table 3), dual implementation shown in Figures 5/8/11 and Table 5, zero-shot probing validated (0.81\u00b10.02, Figure 6). No conclusions contradict or exaggerate results.",
    "CS2_Plan_vs_Implementation": "Section 2 explicitly defines methodology: 2.1 notation, 2.2 filter heads/activation patching, 2.3 DCM with sparse masking. Experimental sections 3.1-3.3 follow this plan exactly: portability within task, across 

In [4]:
# Now copy the notebook to the required location
import shutil

# Source notebook path
source_nb = '/home/smallyan/eval_agent/notebooks/2026-01-14-23-50_self_matching.ipynb'
dest_nb = '/net/scratch2/smallyan/filter_eval/doc_only_evaluation/self_matching.ipynb'

shutil.copy(source_nb, dest_nb)
print(f"Notebook copied to: {dest_nb}")

# Verify both files exist
print("\nVerification:")
print(f"JSON exists: {os.path.exists('/net/scratch2/smallyan/filter_eval/doc_only_evaluation/consistency_evaluation.json')}")
print(f"Notebook exists: {os.path.exists(dest_nb)}")

Notebook copied to: /net/scratch2/smallyan/filter_eval/doc_only_evaluation/self_matching.ipynb

Verification:
JSON exists: True
Notebook exists: True


In [5]:
# List the output directory contents
print("Output directory contents:")
for f in os.listdir('/net/scratch2/smallyan/filter_eval/doc_only_evaluation/'):
    fpath = os.path.join('/net/scratch2/smallyan/filter_eval/doc_only_evaluation/', f)
    print(f"  {f} ({os.path.getsize(fpath)} bytes)")

Output directory contents:
  generalization_eval_summary.json (2033 bytes)
  code_critic_evaluation.ipynb (26423 bytes)
  code_critic_summary.json (1400 bytes)
  consistency_evaluation.json (2327 bytes)
  self_matching.ipynb (13550 bytes)
  generalization_eval.ipynb (15989 bytes)
